# Mapillary-Abdeckung

Welche Stadt, steht in `config.yaml` unter `city` — dieses Notebook liest sie
dort und legt alles unter dem daraus gebildeten Slug ab (`Osnabrück, Germany`
→ `data/osnabrueck/`). Für eine zweite Stadt reicht es, `city` umzustellen;
soll die erste Stadt daneben unangetastet weiterlaufen, sticht `VPR_CITY` die
Datei, ohne sie zu ändern:

```
VPR_CITY="Würzburg, Germany" jupyter lab notebooks/01_mapillary_coverage.ipynb
```

Erfasst alle Mapillary-Bildstandorte im Stadtgebiet, wertet die Abdeckung aus und erzeugt einen reproduzierbaren, sequence-basierten Datensatz für die spätere VPR-Evaluation.

**Warum Vector Tiles und nicht die Bbox-Suche?**
`graph.mapillary.com/images?bbox=` ist eine Suchschnittstelle und antwortet
nachweislich unvollständig — gemessen wurde eine Kachel mit 30 Treffern, die
beim Vierteln 39 ergab. Es gibt keine Kachelgröße und keine Trefferzahl, an der
man das erkennen könnte, und es kommt weder Fehler noch Warnung.
Vector Tiles sind dieselbe Quelle, aus der mapillary.com seine Karte zeichnet:
vollständig per Konstruktion, und für eine Stadt von rund 120 km² reichen
~130 Abfragen.

**Zwei Datenquellen, zwei Gewichtsklassen.** Der Datensatz kommt vollständig
von Mapillary und steht nach Abschnitt 6 fest. Alles danach — Straßennetz,
Stadtteile — kommt von OpenStreetMap über Overpass, einen öffentlichen,
geteilten Dienst, bei dem Zeitüberschreitungen und Sperren Normalbetrieb sind.
Diese Abschnitte fallen im Zweifel mit einer Meldung aus, statt den Lauf
abzubrechen; Endpunkt, Zeitlimit und Ratenbremse stehen in `config.yaml` unter
`osm` (Spiegel je Rechner: `VPR_OVERPASS_URL`).

`.env` im Projektordner mit `MAPILLARY_TOKEN=MLY|...` (Vorlage: `.env.example`).
Alle übrigen Parameter stehen in `config.yaml`.


**Datensatz-Spezifikation:** Die Coverage-Stufe erhält `image_id`, `sequence_id`, `captured_at`, `lat`, `lon`, `compass_angle`, `is_pano` und `creator_id`. Sequenzen werden als unteilbare Einheiten in `train`, `database` und `query` aufgeteilt. Für räumliche Beziehungen gelten 10 m als positiv, 10–25 m als unsicher und >25 m als negativ.

## 1. Konfiguration


In [ ]:
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd


PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, paths
from src.districts import (
    assign_district,
    configure_osmnx,
    geocode_city,
    load_districts,
    overpass_url,
)
from src.mapillary import load_token, make_session
from src.run_guard import validate_config
from src.split import split_column, split_sequences, write_split_lists

# Parameter: versioniert, damit alle im Team denselben Datensatz erzeugen.
CFG = load_config(PROJECT_ROOT)
PATHS = paths(CFG, PROJECT_ROOT)

# Split-Anteile und Radien prueft validate_config beim Laden.
validate_config(CFG)

CITY_NAME   = CFG["city"]
ZOOM        = CFG["zoom"]
MAX_WORKERS = CFG["tile_workers"]

# Reproduzierbare MSLS-inspirierte Datensatz-Spezifikation.
# Sequenzen sind die Split-Einheit; einzelne Bilder werden niemals getrennt.
SPLIT_SEED = int(CFG["vpr"]["split_seed"])

if MAX_WORKERS == "auto":
    MAX_WORKERS = min(16, (os.cpu_count() or 4) * 2)
else:
    MAX_WORKERS = int(MAX_WORKERS)

DISTRICTS_ENABLED = CFG.get("districts", {}).get("enabled", True)

# Token: geheim, deshalb bewusst nicht in config.yaml.
TOKEN = load_token(PROJECT_ROOT)

PROCESSED_DIR = PATHS.processed
RAW_DIR       = PATHS.raw
FIGURE_DIR    = PATHS.figures / "coverage"
CACHE_DIR     = PATHS.cache
for ordner in (RAW_DIR, PROCESSED_DIR, FIGURE_DIR, CACHE_DIR):
    ordner.mkdir(parents=True, exist_ok=True)

F_TILES = RAW_DIR / "vt_tiles.json"     # Rohpunkte je Kachel, dient als Checkpoint
F_ALL   = RAW_DIR / "vt_images.csv"     # alle Punkte im Kachelbereich
F_CITY  = RAW_DIR / "images_city.csv"   # Endergebnis: nur innerhalb der Stadtgrenze
F_METADATA = PROCESSED_DIR / "metadata.parquet"

# Cache und Overpass-Einstellungen (Endpunkt, Zeitlimit, Ratenbremse) aus
# config.yaml -> osm, einmal fuer alle osmnx-Aufrufe dieses Laufs.
configure_osmnx(CFG, CACHE_DIR)
plt.rcParams["figure.dpi"] = 300

print(f"{CITY_NAME}  ->  data/{PATHS.city}/")
print(f"Zoom {ZOOM} | Stadtteile {'✓' if DISTRICTS_ENABLED else '–'} | "
      f"Token ✓ | Konfig: config.yaml")
print(f"CPU-Threads {os.cpu_count()} | Download-Worker {MAX_WORKERS}")
print(f"Overpass    {overpass_url()}")

## 2. Hilfsfunktionen

Kachelmathematik (Web-Mercator wie OSM), Dekodierung der Kacheln und die
HTTP-Sitzung mit Wiederholung stehen in `src/mapillary.py`.

In [ ]:
from src.mapillary import load_tile, tiles_for_bounds

## 3. Stadtgebiet und Kachelbereich


In [ ]:
# Stadtgrenze als Polygon -- dieselbe Auswahl wie in src/districts.py, damit
# 01 und die Experimente garantiert dieselbe Stadt meinen. Der Geocoder
# liefert je nach Suchbegriff mehrere Objekte (Punkt, Stadt, Landkreis);
# geocode_city filtert auf Flaechen und nimmt das erste.
#
# Bei einer neuen Stadt lohnt der Blick auf Flaeche und osm_id: liegt die
# Flaeche weit ueber dem Stadtgebiet, hat der Geocoder den gleichnamigen
# Landkreis geliefert -- dann city praeziser angeben ("Stadt X, Bundesland,
# Germany"). Zum Vergleich: Osnabrueck 119,7 km2, Wuerzburg 87,6 km2, der
# Landkreis Wuerzburg dagegen knapp 1.000 km2.
city_polygon, UTM_CRS, city_row = geocode_city(CITY_NAME)
city_area = gpd.GeoSeries([city_polygon], crs="EPSG:4326").to_crs(UTM_CRS).area.iloc[0] / 1e6

lon_min, lat_min, lon_max, lat_max = city_polygon.bounds
TILES = tiles_for_bounds(lon_min, lat_min, lon_max, lat_max, ZOOM)

print(city_row.get("display_name", CITY_NAME))
print(f"OSM {city_row.get('osm_type', '?')}/{city_row.get('osm_id', '?')}  "
      f"{city_row.get('class', '?')}={city_row.get('type', '?')}")
print(f"Fläche: {city_area:.1f} km²   Kacheln: {len(TILES)}   {UTM_CRS.name}")

## 4. Kacheln laden

Wie viele Kacheln, steht in der Ausgabe oben; der Lauf dauert wenige Minuten.
Die Rohpunkte landen in `data/<stadt>/raw/vt_tiles.json` und werden während
des Laufs regelmäßig fortgeschrieben — ein zweiter Lauf liest sie von dort und
holt nur, was fehlt. Ein bewusster Neuaufbau löscht die Datei.

Eine Kachel, die sich auch nach den Wiederholungen aus `make_session()` nicht
laden lässt, bricht die Zelle ab. Das ist Absicht: sie wäre ein fehlendes Stück
Stadt, und der Split direkt darunter wird aus genau diesen Daten gezogen und in
`*_sequences.txt` **eingefroren**. Ein stiller Verlust hier wäre später an
keiner Zahl mehr zu erkennen.

In [ ]:
# Die Rohpunkte je Kachel bleiben als Datei liegen: ein zweiter Lauf laedt
# nichts neu, ein bewusster Neuaufbau loescht vt_tiles.json.
done = json.loads(F_TILES.read_text()) if F_TILES.exists() else {}
todo = [t for t in TILES if f"{t[0]}_{t[1]}" not in done]
print(f"{len(todo)} von {len(TILES)} Kacheln offen")


def fetch_tile(tile):
    # Eine Session pro Worker-Aufruf vermeidet offene Verbindungen und nutzt
    # trotzdem die Retry-/Connection-Pool-Konfiguration aus make_session(MAX_WORKERS * 4).
    session = make_session(MAX_WORKERS * 4)
    try:
        return tile, load_tile(session, TOKEN, ZOOM, *tile)
    finally:
        session.close()


failed_tiles = []

if todo:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        jobs = {pool.submit(fetch_tile, t): t for t in todo}
        for i, job in enumerate(as_completed(jobs), 1):
            try:
                (tx, ty), pts = job.result()
                done[f"{tx}_{ty}"] = pts
            except Exception as e:
                tile = jobs[job]
                failed_tiles.append(tile)
                print(f"\n  Kachel {tile} fehlgeschlagen: {type(e).__name__}: {e}")
            # Zwischenstand sichern. Ein abgestuerzter Kernel oder ein mitten
            # im Lauf ablaufendes Token kostete sonst alle bereits geholten
            # Kacheln; die Datei zu schreiben dauert Sekunden, deshalb nicht
            # nach jeder einzelnen.
            if i % 25 == 0:
                F_TILES.write_text(json.dumps(done))
            if i % 10 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)} Kacheln — "
                      f"{sum(len(v) for v in done.values()):,} Punkte "
                      f"({time.time() - t0:.0f}s)", end="\r")
    F_TILES.write_text(json.dumps(done))

# Eine leere Kachel ist kein Fehler (src/mapillary.load_tile gibt [] zurueck,
# auch bei 404). Was hier landet, hat die fuenf Wiederholungen aus
# make_session() ueberlebt -- also ein echter Ausfall und damit ein fehlendes
# Stueck Stadt. Der Split unten wird aus diesen Daten gezogen und eingefroren;
# lieber hier abbrechen als eine Zahl, der niemand ansieht, dass sie falsch ist.
if failed_tiles:
    beispiele = ", ".join(f"{x}_{y}" for x, y in failed_tiles[:5])
    raise RuntimeError(
        f"{len(failed_tiles)} von {len(TILES)} Kacheln nicht geladen ({beispiele}"
        f"{', ...' if len(failed_tiles) > 5 else ''}).\n"
        f"Alles bereits Geholte steht in {F_TILES.name} -- diese Zelle einfach "
        "noch einmal ausfuehren, sie holt genau die fehlenden nach."
    )

rows = [r for v in done.values() for r in v]
if not rows:
    raise RuntimeError("Keine Mapillary-Bilder aus den Vector Tiles geladen.")

images = pd.DataFrame(rows)
required = {"image_id", "sequence_id", "lon", "lat"}
missing = required - set(images.columns)
if missing:
    raise RuntimeError(
        f"Mapillary-Daten enthalten nicht alle benötigten Spalten: {sorted(missing)}"
    )

# Einheitliches Metadaten-Schema. Optional fehlende Tile-Properties bleiben als NA
# erhalten; dadurch bleibt die Datei auch bei unterschiedlichen Tile-Versionen stabil.
for col in ["captured_at", "compass_angle", "is_pano", "creator_id"]:
    if col not in images.columns:
        images[col] = pd.NA

images = images[
    ["image_id", "sequence_id", "captured_at", "lat", "lon",
     "compass_angle", "is_pano", "creator_id"]
].drop_duplicates(subset="image_id").reset_index(drop=True)

images.to_csv(F_ALL, index=False)

n_sequences = images["sequence_id"].nunique()
print(f"\n\n{len(images):,} Punkte, {n_sequences:,} Sequenzen -> {F_ALL}")

## 5. Auf das Stadtgebiet schneiden


In [ ]:
images_gdf = gpd.GeoDataFrame(
    images, geometry=gpd.points_from_xy(images["lon"], images["lat"]), crs="EPSG:4326")

# covered_by schliesst Punkte auf der Stadtgrenze ein. Das ist fuer GPS-Punkte
# sinnvoller als within, das Randpunkte ausschliesst.
images_gdf = images_gdf[images_gdf.geometry.covered_by(city_polygon)].reset_index(drop=True)
images_gdf.drop(columns="geometry").to_csv(F_CITY, index=False)

pct_city = len(images_gdf) / len(images) * 100 if len(images) else 0.0
print(f"Im Stadtgebiet: {len(images_gdf):,} von {len(images):,} ({pct_city:.1f} %)")
print(f"Dichte: {len(images_gdf) / city_area:,.0f} Bilder/km²   -> {F_CITY}")

## 6. Reproduzierbarer VPR-Datensatz

Die Coverage-Erhebung endet nicht bei einer reinen Bildzählung. Gemäß Spezifikation
werden die Metadaten als reproduzierbarer Datensatz gespeichert. Die Split-Einheit ist
`sequence_id`: Eine Sequenz liegt vollständig in genau einem von `train`, `database` oder
`query`.

Zusätzlich werden für die räumliche Evaluation GPS-Kandidaten erzeugt:
- **≤ 10 m:** positive candidate
- **10–25 m:** uncertain / nicht als sicher negativ verwenden
- **> 25 m:** negative candidate

Die Paare werden nur zwischen `query` und `database` erzeugt. Damit kann die spätere
Retrieval-Evaluation direkt auf denselben, unveränderten Splits aufbauen.


In [ ]:
# Sequence-aware Split und VPR-Metadaten.
# Wichtig: Wir splitten ausschließlich sequence_id, niemals einzelne Bilder.

if images_gdf.empty:
    raise RuntimeError("Keine Bilder innerhalb der Stadtgrenze.")

metadata = images_gdf.drop(columns="geometry").copy()

# Die Split-Logik steht in src/split.py und ist dort getestet: vorhandene
# Listen werden uebernommen (die Mapillary-Daten aendern sich, derselbe Seed
# ergaebe sonst einen anderen Split und alle Embeddings waeren wertlos),
# sonst wird mit dem Seed gewuerfelt.
train_sequences, database_sequences, query_sequences, herkunft = split_sequences(
    metadata["sequence_id"].dropna().astype(str).unique(), PROCESSED_DIR, CFG
)
print(f"Split {herkunft} (seed={SPLIT_SEED}).")
metadata["split"] = split_column(
    metadata["sequence_id"].astype(str), train_sequences, database_sequences, query_sequences
)

ohne_split = int(metadata["split"].isna().sum())
if ohne_split == len(metadata):
    raise RuntimeError(
        "Keinem einzigen Bild konnte ein Split zugeordnet werden -- "
        "Sequenzlisten und Daten passen nicht zusammen."
    )
if ohne_split:
    print(f"{ohne_split:,} Bilder ohne Split-Zuordnung werden verworfen.")
    metadata = metadata[metadata["split"].notna()].reset_index(drop=True)
    # Karten und Bericht weiter unten sollen denselben Datensatz zeigen, den
    # 02 bis 08 rechnen -- nicht die paar Bilder mehr, die es vor dem Split
    # gab. Sonst zaehlt der Bericht mehr Bilder, als die Split-Zeile darunter
    # zusammen ausweist.
    images_gdf = images_gdf[
        images_gdf["image_id"].isin(set(metadata["image_id"]))
    ].reset_index(drop=True)

# Reproduzierbare Artefakte. Die Kacheln kommen aus einem Threadpool in
# Ankunftsreihenfolge; sortiert ist die Datei auf jedem Rechner gleich.
# Betrifft nur frische Laeufe -- die versionierte Parquet bleibt, wie sie
# ist, sonst passen die Fingerabdruecke der Embeddings nicht mehr.
metadata = metadata.sort_values(["sequence_id", "captured_at", "image_id"])
metadata = metadata.reset_index(drop=True)
metadata.to_parquet(F_METADATA, index=False)
write_split_lists(PROCESSED_DIR, train_sequences, database_sequences, query_sequences)

print("Sequence-aware Split:")
print(f"  train:    {len(train_sequences):,} Sequenzen / {(metadata['split'] == 'train').sum():,} Bilder")
print(f"  database: {len(database_sequences):,} Sequenzen / {(metadata['split'] == 'database').sum():,} Bilder")
print(f"  query:    {len(query_sequences):,} Sequenzen / {(metadata['split'] == 'query').sum():,} Bilder")
print(f"  Seed:     {SPLIT_SEED}")
print(f"  -> {F_METADATA}")

# Kandidatenpaare (Query -> Datenbank, Anchor -> Positive) entstehen nicht
# mehr hier: 02 und 05 leiten sie in Sekunden aus den Metadaten ab
# (src/pairs.py), nichts davon muss versioniert werden.

# Konsistenzprüfungen: Keine Sequenz darf in mehreren Splits vorkommen.
assert not (
    set(train_sequences) & set(database_sequences)
    or set(train_sequences) & set(query_sequences)
    or set(database_sequences) & set(query_sequences)
), "Sequence leakage im Split!"


## 7. Stadtweite Karte

Ab hier kommen die Daten von OpenStreetMap statt von Mapillary. Der Datensatz
oben steht bereits auf der Platte — das Straßennetz ist nur Hintergrund. Fällt
Overpass aus, entsteht die Karte ohne Straßen, und der Lauf geht weiter.

In [ ]:
# network_type="drive" reicht als Hintergrund und lädt deutlich schneller
# als "all" (das zusätzlich Fuß- und Radwege enthält).
#
# Einzige Overpass-Abfrage, die 01 fuer eine Abbildung braucht -- und die
# schwerste im Notebook: das Fahrnetz einer ganzen Stadt. An einem
# ausgelasteten Endpunkt kommt hier ein Timeout oder ein 504. Das darf den
# Lauf nicht kosten: metadata.parquet und die Split-Listen sind oben
# geschrieben, run.py wuerde sonst mitten in einer fertigen Stufe abbrechen.
try:
    street_net = ox.graph_from_polygon(city_polygon, network_type="drive")
except Exception as e:
    street_net = None
    print(f"Straßennetz nicht geladen ({type(e).__name__}: {e})\n"
          "Karte ohne Straßen. Mit einem Spiegel erneut versuchen:\n"
          '  VPR_OVERPASS_URL="https://overpass.kumi.systems/api"')

if street_net is not None:
    fig, ax = ox.plot_graph(street_net, node_size=0, edge_linewidth=0.35,
                            edge_color="#cccccc", bgcolor="white",
                            show=False, close=False, figsize=(14, 14))
else:
    # ox.plot_graph setzt fuer EPSG:4326 selbst das Seitenverhaeltnis nach
    # cos(Breitengrad); ohne Graph muessen wir das hier tun.
    fig, ax = plt.subplots(figsize=(14, 14))
    ax.set_facecolor("white")
    ax.set_aspect(1 / np.cos(np.radians(city_polygon.centroid.y)))
    ax.set_axis_off()

ax.scatter(images_gdf["lon"], images_gdf["lat"], s=0.8, c="#2ca02c",
           alpha=0.35, edgecolors="none", zorder=5)
gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
    ax=ax, color="black", linewidth=1.2, linestyle="--")
ax.set_title(f"{CITY_NAME.split(',')[0]} — {len(images_gdf):,} Mapillary-Bilder")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "coverage_city.png"); plt.show()

## 8. Stadtteile bestimmen

Die gesamte Stadtteilberechnung kann in `config.yaml` mit `districts.enabled` aktiviert oder deaktiviert werden.

Die administrative Gliederung ist die Primaerquelle. Zuerst werden `boundary=administrative`-Flaechen innerhalb der bereits bekannten Stadtgrenze geladen und nur die konfigurierten `admin_level`-Werte betrachtet. Standardmaessig wird `admin_level=10` vor `admin_level=9` bevorzugt.

Die Auswahl prueft:
1. **Tatsaechliche Coverage** — Union der Polygone statt einfacher Flächensumme.
2. **Ueberlappung** — doppelt bedeckte Flaechen werden als Qualitaetsproblem erkannt.
3. **Mindestanzahl und Mindestflaeche** — sehr kleine oder unbrauchbare Kandidaten werden entfernt.
4. **Clippen an der Stadtgrenze** — die zurueckgegebenen Polygone liegen wirklich im Stadtgebiet.
5. **place-Tags als Fallback** — nur wenn keine brauchbare administrative Ebene vorhanden ist.

Damit werden unterschiedliche Hierarchieebenen nicht mehr anschliessend per Geometrieheuristik vermischt.

In [ ]:
# Die Auswahl der Ebene (admin_level aus config.yaml -> districts, Wertung
# Abdeckung minus Ueberlappung, Clipping, Mindestflaeche) steht in
# src/districts.py -- dieselbe Gliederung nutzen recall_by_district.py und
# confusion_atlas.py.
#
# city_polygon und UTM_CRS reichen wir durch: das spart einen zweiten
# Geocode und stellt sicher, dass die Stadtteile an genau dieselbe Grenze
# geclippt werden, gegen die oben die Bilder gefiltert wurden.
if not DISTRICTS_ENABLED:
    districts_gdf = None
    print("Stadtteil-Auswertung deaktiviert (districts.enabled: false).")
else:
    try:
        districts_gdf, _, _ = load_districts(CFG, CACHE_DIR, city_polygon, UTM_CRS)
    except Exception as e:
        # RuntimeError = keine brauchbare Gliederung (echtes Ergebnis fuer
        # manche Staedte). Alles andere kommt von Overpass: Timeout, 429,
        # 504. Beides ist kein Grund, den fertigen Datensatz von oben
        # wegzuwerfen -- die Stadtteilabschnitte entfallen und 01 laeuft zu
        # Ende. Nachholen heisst spaeter: dieses Notebook noch einmal
        # ausfuehren. Der Split wird dabei nicht neu gewuerfelt, sondern aus
        # den *_sequences.txt uebernommen (src/split.py).
        districts_gdf = None
        print(f"Keine Stadtteile ({type(e).__name__}): {e}")

HAS_DISTRICTS = districts_gdf is not None
if HAS_DISTRICTS:
    print(f"{len(districts_gdf)} Stadtteile:")
    for _, row in districts_gdf.sort_values("name").iterrows():
        print(f"  {row['name']:32s} {row['area_km2']:6.2f} km²")
    metric = districts_gdf.to_crs(UTM_CRS)
    union_area = metric.geometry.union_all().area / 1e6
    print(f"\nStadtteilabdeckung: {union_area:.1f} km² ({union_area / city_area:.0%} der Stadtflaeche)")

## 9. Abdeckung pro Stadtteil


In [ ]:
if not HAS_DISTRICTS:
    metrics_df = None
    print("Uebersprungen: keine Stadtteile.")
else:
    # Punkt-in-Polygon je Bild; bei Mehrfachtreffern die kleinste Flaeche.
    images_gdf["district"] = assign_district(
        images_gdf["lat"].to_numpy(), images_gdf["lon"].to_numpy(), districts_gdf
    )
    n_unassigned = int(images_gdf["district"].isna().sum())
    print(f"Ohne Stadtteil-Zuordnung: {n_unassigned:,} ({n_unassigned / len(images_gdf):.1%})")
    if n_unassigned / len(images_gdf) > 0.2:
        print("Ueber 20 % unzugeordnet -- die Polygone decken das Stadtgebiet nur lueckenhaft ab.")

    je_bezirk = images_gdf.groupby("district").agg(
        Bilder=("image_id", "size"), Sequenzen=("sequence_id", "nunique")
    )
    metrics_df = districts_gdf[["name", "area_km2"]].rename(
        columns={"name": "Stadtteil", "area_km2": "Fläche km²"}
    ).join(je_bezirk, on="Stadtteil").fillna({"Bilder": 0, "Sequenzen": 0})
    metrics_df["Bilder/km²"] = (metrics_df["Bilder"] / metrics_df["Fläche km²"]).round()
    metrics_df = metrics_df.sort_values("Bilder/km²", ascending=False).reset_index(drop=True)

    median = metrics_df["Bilder/km²"].median()
    print(f"Median: {median:,.0f} Bilder/km²")
    schwach = metrics_df[metrics_df["Bilder/km²"] < median * 0.5]
    if len(schwach):
        print(f"{len(schwach)} Stadtteile unter der Haelfte des Medians:")
        for _, r in schwach.iterrows():
            print(f"   {r['Stadtteil']:<32} {r['Bilder/km²']:>8,.0f}")

metrics_df


## 10. Choroplethenkarte


In [ ]:
if not HAS_DISTRICTS:
    print("Übersprungen: braucht Stadtteile.")
else:
    plot_gdf = districts_gdf.merge(metrics_df, left_on="name",
                                   right_on="Stadtteil", how="left")

    fig, ax = plt.subplots(figsize=(13, 11))
    plot_gdf.plot(column="Bilder/km²", cmap="YlOrRd", legend=True, ax=ax,
                  edgecolor="black", linewidth=0.5,
                  legend_kwds={"label": "Bilder pro km²", "shrink": 0.7},
                  missing_kwds={"color": "lightgrey", "label": "keine Daten"})

    for _, r in plot_gdf.iterrows():
        p = r.geometry.representative_point()      # liegt garantiert im Polygon
        ax.annotate(r["name"], xy=(p.x, p.y), fontsize=7, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.65, ec="none"))

    gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
        ax=ax, color="black", linewidth=1.5, linestyle="--")
    ax.set_aspect(1 / np.cos(np.radians(city_polygon.centroid.y)))
    ax.set_title(f"Mapillary-Abdeckung nach Stadtteil — {CITY_NAME.split(',')[0]}")
    plt.tight_layout(); plt.savefig(FIGURE_DIR / "choropleth_coverage.png"); plt.show()


## 11. Dichte-Heatmap

Geglättetes 2D-Histogramm statt Kerndichteschätzung: eine echte KDE müsste für
jeden Rasterpunkt über alle Bilder summieren — bei den mehreren Hunderttausend
je Stadt läuft das praktisch nicht mehr. Das Ergebnis sieht gleich aus und
rechnet in Sekunden.

In [ ]:
from scipy.ndimage import gaussian_filter

pad_x = (lon_max - lon_min) * 0.03
pad_y = (lat_max - lat_min) * 0.03
extent = [lon_min - pad_x, lon_max + pad_x, lat_min - pad_y, lat_max + pad_y]

density, _, _ = np.histogram2d(images_gdf["lon"], images_gdf["lat"],
                               bins=[420, 300],
                               range=[extent[:2], extent[2:]])
density = gaussian_filter(density, sigma=2.0)

fig, ax = plt.subplots(figsize=(13, 11))
# log1p, weil die Innenstadt sonst alles andere überstrahlt
im = ax.imshow(np.log1p(density).T, origin="lower", extent=extent,
               cmap="inferno", interpolation="bilinear", aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.7, label="Bilddichte (logarithmisch)")

gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
    ax=ax, color="white", linewidth=1.5, linestyle="--")
ax.set_aspect(1 / np.cos(np.radians(city_polygon.centroid.y)))
ax.set_title(f"Bilddichte — {CITY_NAME.split(',')[0]} ({len(images_gdf):,} Bilder)")
plt.tight_layout(); plt.savefig(FIGURE_DIR / "heatmap_coverage.png"); plt.show()


## 12. Bericht


In [ ]:
per_seq = images_gdf["sequence_id"].value_counts() if "sequence_id" in images_gdf.columns else pd.Series(dtype="int64")
captured = pd.to_datetime(images_gdf["captured_at"], unit="ms", errors="coerce")
valid_captured = captured.dropna()

print("=" * 58)
print(f"  Stadt                {CITY_NAME}")
print(f"  Fläche               {city_area:,.1f} km²")
print(f"  Bilder               {len(images_gdf):,}")
print(f"  Dichte               {len(images_gdf) / city_area:,.0f} /km²")
print(f"  Sequenzen            {per_seq.size:,}")
if len(per_seq):
    print(f"  Median je Sequenz    {per_seq.median():,.0f}   (längste: {per_seq.max():,})")
if len(valid_captured):
    print(f"  Aufnahmezeitraum     {valid_captured.min():%Y-%m} bis {valid_captured.max():%Y-%m}")
else:
    print("  Aufnahmezeitraum     nicht verfügbar")
print(f"  Stadtteile           {len(districts_gdf) if HAS_DISTRICTS else '– (keine)'}")
print(f"  Straßennetz          {'✓' if street_net is not None else '– (Overpass)'}")
print("=" * 58)

print(f"  Split                train={len(train_sequences):,} / database={len(database_sequences):,} / query={len(query_sequences):,}")
print(f"  Ground-Truth-Radius  <= {CFG['vpr']['uncertain_radius_m']:g} m")
print(f"  Metadaten            {F_METADATA}")

# Die Stufe ist fertig, sobald metadata.parquet und die Split-Listen stehen --
# run.py prueft genau das und geht weiter zu 02. Fehlende Karten oder
# Stadtteile heissen nur, dass Overpass nicht mitgespielt hat; nachholen
# laesst sich das jederzeit, indem man dieses Notebook noch einmal
# ausfuehrt. Am Datensatz aendert sich dabei nichts: die Split-Listen werden
# uebernommen, nicht neu gewuerfelt.
if not HAS_DISTRICTS or street_net is None:
    print("\nHinweis: OSM-Teile unvollständig — der Datensatz ist davon nicht betroffen.")